[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C59_VLA_Perception_Interface_Course/05_eval_deploy/05_vla_eval_deploy.ipynb)

# 05 · VLA 的评测与上车（开环vs闭环 / 指标体系 / 幻觉度量 / 蒸馏一致性 / 云端分配）

目标：把「这个模型好不好」这个问题，拆成**一组各自有分母、各自有作弊路径、各自有门禁**的可计算指标。

本 notebook 你会亲手实现：
1. **开环 vs 闭环的差异模拟**：两个开环 ADE 几乎相同的模型，闭环失败率差 **80 倍**
2. **自车状态捷径**：一个**完全不看图像**的常速外推基线，在开环 ADE 上击败真的在感知的模型
3. **驾驶指标计算器** + CARLA 式**乘法**复合分，并演示「停着不动」这条作弊路径怎么被堵死
4. **幻觉度量**：感知幻觉 / 属性幻觉 / 漏报的分解，以及**诱导式 prompt 把幻觉率拉高一个数量级**
5. **置信度校准 ECE** 与朴素重校准
6. **「多少里程才能证明安全」的泊松功效计算** —— 一个把「多跑路测」这条路堵死的算术
7. **蒸馏一致性**：整体一致率 94% 而长尾桶只有 62%；以及「一致 ≠ 正确」
8. **云端-车端任务分配求解器**（延迟预算 + 截止时间 + 安全关键约束）

> 心智模型：**每一个指标都要问两件事——它的分母是什么，它能被什么方式作弊。**

## 1 · 开环 vs 闭环：同一个模型，两个世界

一维车道保持任务。专家策略 `a* = -0.5·x`（x 是横向偏移），过程噪声 σ=0.10 m。

学到的策略在**训练分布内**（`|x| <= x_train`）忠实复制专家，在**分布外**能力线性衰减到 0
—— 这正是行为克隆的真实行为：训练数据里没有「车已经偏了 0.5 m」的样本，因为专家从不那样开。

两个模型：
- **A（纯行为克隆）**：`x_train = 0.12` —— 只见过专家分布
- **B（加了分布外恢复数据，DAgger 式）**：`x_train = 0.20`，衰减更缓

In [ ]:
import numpy as np, math, itertools

K_EXPERT, EPS, W = 0.5, 0.05, 0.10       # 专家回正增益 / 动作噪声 / 过程噪声

def expert(x):
    return -K_EXPERT * x

def make_policy(x_train, decay):
    """训练分布内忠实复制专家，分布外能力线性衰减到 0。"""
    def fidelity(x):
        return 1.0 if abs(x) <= x_train else max(0.0, 1.0 - decay * (abs(x) - x_train))
    return fidelity

def expert_traj(T, r):
    x, xs = 0.0, []
    for _ in range(T):
        xs.append(x)
        x = x + expert(x) + r.normal(0, W)
    return np.array(xs)

def open_loop_ade(fid, T=300, n_ep=100):
    """开环：**每一步都从专家的真实状态出发**，误差互相独立，不传播。"""
    errs = []
    for e in range(n_ep):
        xs = expert_traj(T, np.random.default_rng(1000 + e))
        rp = np.random.default_rng(5000 + e)
        for x in xs:
            a_hat = expert(x) * fid(x) + rp.normal(0, EPS)
            errs.append(abs(a_hat - expert(x)))
    return float(np.mean(errs))

def closed_loop(fid, T=300, n_ep=200, fail_thr=1.0):
    """闭环：**模型自己的动作决定下一个状态**。偏离 > 1.0 m 判为出车道 = 失败。"""
    fails, devs = 0, []
    for e in range(n_ep):
        r = np.random.default_rng(9000 + e)
        x, failed = 0.0, False
        for _ in range(T):
            x = x + expert(x) * fid(x) + r.normal(0, EPS) + r.normal(0, W)
            devs.append(abs(x))
            if abs(x) > fail_thr:
                failed = True; break
        fails += failed
    return fails / n_ep, float(np.mean(devs))

A = make_policy(0.12, 3.0)     # 纯行为克隆
B = make_policy(0.20, 2.0)     # 加了分布外恢复数据

oa, ob = open_loop_ade(A), open_loop_ade(B)
(fa, da), (fb, db) = closed_loop(A), closed_loop(B)

print(f"{'模型':<28s}{'开环 ADE (m)':>14s}{'闭环平均偏离 (m)':>18s}{'闭环失败率':>12s}")
print('%-28s%14.4f%18.4f%11.1f%%' % ('A · 纯行为克隆', oa, da, 100 * fa))
print('%-28s%14.4f%18.4f%11.1f%%' % ('B · 加分布外恢复数据', ob, db, 100 * fb))
print('\n开环差距：%.1f%%（几乎分不开）      闭环失败率差距：**%.0f 倍**'
      % (100 * abs(oa - ob) / ob, fa / max(fb, 1e-9)))

assert abs(oa - ob) / ob < 0.10, '两个模型在开环上几乎无法区分'
assert fa > 20 * fb, '闭环上却差着一个数量级以上'
assert fa > 0.2 and fb < 0.05
print('\n⚠️ 如果你按开环 ADE 选型，你会认为 A 和 B 一样好 —— 然后把 A 送上车。')

In [ ]:
# —— 误差累积的闭式模型：e_{t+1} = (1+kappa)e_t + eps ——
def error_growth(eps, kappa, T):
    return eps * T if kappa == 0 else eps * ((1 + kappa) ** T - 1) / kappa

print(f"{'步数 T':>8s}{'开环误差 (=eps)':>16s}{'闭环误差 kappa=0.05':>21s}{'kappa=0.10':>14s}{'放大倍数':>10s}")
for T in (1, 5, 10, 20, 30, 50):
    e5, e10 = error_growth(0.05, 0.05, T), error_growth(0.05, 0.10, T)
    print('%8d%16.3f%21.3f%14.3f%9.0fx' % (T, 0.05, e5, e10, e10 / 0.05))
assert abs(error_growth(0.05, 0.10, 30) / 0.05 - 164.49) < 0.1
assert error_growth(0.05, 0.0, 30) == 0.05 * 30, 'kappa=0 时退化成线性累积'
print('\n✅ kappa=0.10、T=30 时闭环误差是开环报数的 **164 倍**：开环 5 cm，闭环 8.2 m。')
print('   模仿学习的理论界是 O(eps·T²) 而不是 O(eps·T) —— 正是这个反馈回路造成的。')

### 1.2 自车状态捷径：不看图像也能赢

开环评测通常把「自车过去几秒的位置与速度」也喂给模型。而人类驾驶极其平滑，
**过去 2 秒的状态几乎决定了未来 3 秒的轨迹**。

下面构造两个「模型」：
- `M_ego`：**完全不看图像**，常速直线外推（等价于预测曲率 = 0）
- `M_vision`：真的在估计前方道路曲率，但有噪声

数据：4% 的路段是弯道（曲率 0.003–0.012 /m，即半径 83–333 m），其余是直路。
车速 15 m/s，预测时域 3 s。横向位移 $y(t)=\tfrac12\kappa v^2 t^2$。

In [ ]:
rng2 = np.random.default_rng(3)
N_SEG, V, H = 8000, 15.0, 3.0

is_curve = rng2.random(N_SEG) < 0.04
kappa_gt = np.where(is_curve, rng2.uniform(0.003, 0.012, N_SEG), 0.0)
kappa_ego = np.zeros(N_SEG)                                # 常速直线外推：曲率恒为 0
kappa_vis = kappa_gt + rng2.normal(0, 0.0005, N_SEG)       # 真的在估曲率，但有噪声

ade = lambda dk: 0.5 * np.abs(dk) * V * V * H * H / 3.0    # 平均位移误差
fde = lambda dk: 0.5 * np.abs(dk) * V * V * H * H          # 终点位移误差

e_ego, e_vis = ade(kappa_gt - kappa_ego), ade(kappa_gt - kappa_vis)
f_ego, f_vis = fde(kappa_gt - kappa_ego) > 1.5, fde(kappa_gt - kappa_vis) > 1.5   # 偏 1.5 m = 出车道

print(f"{'切片':<16s}{'样本数':>8s}{'M_ego ADE':>12s}{'M_vision ADE':>14s}{'谁赢':>10s}")
for name, m in [('**整体**', np.ones(N_SEG, bool)), ('直路 (96%)', ~is_curve), ('**弯道 (4%)**', is_curve)]:
    a, b = e_ego[m].mean(), e_vis[m].mean()
    print('%-16s%8d%12.4f%14.4f%10s' % (name, m.sum(), a, b, 'M_ego' if a < b else 'M_vision'))
print('\n闭环（3 s 后横向偏离 > 1.5 m 判为出车道）失败率：'
      'M_ego %.2f%%   M_vision %.2f%%   -> **相差 %.0f 倍**'
      % (100 * f_ego.mean(), 100 * f_vis.mean(), f_ego.mean() / max(f_vis.mean(), 1e-9)))

assert e_ego.mean() < e_vis.mean(), '不看图像的基线在整体开环 ADE 上更好'
assert e_ego[is_curve].mean() > 10 * e_vis[is_curve].mean(), '但在弯道桶上差一个数量级'
assert f_ego.mean() > 5 * f_vis.mean()
print('\n✅ **必做的对照实验：ego-state ablation** —— 把自车历史从输入里去掉，看指标掉多少。')
print('   若一个纯自车基线能达到你模型 90% 的分数，这个 benchmark 在这个任务上就没有分辨力。')

## 2 · 指标体系：六类指标与「防作弊」的乘法结构

只优化安全指标有一条完美的作弊路径：**停在原地不动**。
所以必须有进度类指标做乘数。CARLA Leaderboard 的
$\text{DS}=\text{RC}\times\prod_i p_i^{n_i}$ 正是为此设计的。

In [ ]:
PENALTY = dict(collision_vehicle=0.60, collision_layout=0.65,
               red_light=0.70, stop_sign=0.80, speeding=0.90)

def driving_score(rc, infractions, penalty=None):
    """CARLA 式复合分：**完成率是乘数，违规是折扣**。"""
    p = penalty or PENALTY
    ds = float(rc)
    for k, n in infractions.items():
        ds *= p[k] ** n
    return ds

def rollout_metrics(v, dt, v_limit, route_len):
    d = np.cumsum(v) * dt
    dist = float(d[-1])
    over = v > v_limit + 1e-9
    a = np.diff(v) / dt
    jerk = np.diff(a) / dt if a.size > 1 else np.zeros(1)
    return dict(dist_m=dist, rc=min(1.0, dist / route_len),
                overspeed_ratio=float(np.sum(v[over]) * dt) / max(dist, 1e-9),
                a_p95=float(np.percentile(np.abs(a), 95)),
                jerk_p95=float(np.percentile(np.abs(jerk), 95)))

DT, T_STEP, ROUTE, V_LIMIT = 0.1, 600, 900.0, 60 / 3.6      # 60 s，900 m 路线，限速 60 km/h
r4 = np.random.default_rng(5)
DRIVERS = {
    '激进（快但撞了一次）': (np.clip(19.0 + np.cumsum(r4.normal(0, 0.12, T_STEP)), 0, None),
                             dict(collision_vehicle=1, speeding=1)),
    '极度保守（几乎不动）': (np.clip(2.5 + np.cumsum(r4.normal(0, 0.02, T_STEP)), 0, None),
                             dict()),
    '均衡':                 (np.clip(15.0 + np.cumsum(r4.normal(0, 0.05, T_STEP)), 0, None),
                             dict()),
}

print(f"{'驾驶风格':<22s}{'里程 m':>9s}{'完成率':>8s}{'超速比':>8s}{'jerk p95':>10s}"
      f"{'碰撞':>6s}{'**Driving Score**':>19s}")
res = {}
for name, (v, infr) in DRIVERS.items():
    m = rollout_metrics(v, DT, V_LIMIT, ROUTE)
    ds = driving_score(m['rc'], infr)
    res[name] = (m, infr, ds)
    print('%-22s%9.0f%8.2f%8.1f%%%10.2f%6d%19.3f'
          % (name, m['dist_m'], m['rc'], 100 * m['overspeed_ratio'], m['jerk_p95'],
             infr.get('collision_vehicle', 0), ds))

cons = res['极度保守（几乎不动）']
best = max(res, key=lambda k: res[k][2])
print('\n⚠️ 「极度保守」在**所有安全指标上都是满分**：0 碰撞、0 超速、jerk 最低。')
print('   但它的完成率只有 %.2f -> Driving Score 只有 %.3f。' % (cons[0]['rc'], cons[2]))
print('✅ 复合分最高的是「%s」—— **乘法结构堵死了「不开就不会错」这条作弊路径**。' % best)
assert cons[1] == {} and cons[0]['overspeed_ratio'] == 0.0
assert best == '均衡'
assert res['激进（快但撞了一次）'][2] < res['均衡'][2]
assert abs(res['激进（快但撞了一次）'][2] - 1.0 * 0.60 * 0.90) < 1e-9

## 3 · 幻觉度量：三种形态必须分开报

把每帧「模型报告看到的标志集合」与 GT 集合做匹配，分解成四类：

| 类别 | 定义 | 下游后果 |
|---|---|---|
| **exact** | 类型与数值都对 | 正常 |
| **属性幻觉** | 类型对、数值错（60 说成 80） | **约束值错** |
| **感知幻觉** | 报了一个 GT 里连类型都没有的标志 | **凭空多一条约束** |
| **漏报** | GT 有但没报 | **缺约束** |

关键：评测集里必须有**「无标志帧」这类负样本**，否则幻觉率会被系统性低估。

In [ ]:
SPEED_VALUES = [30, 40, 50, 60, 80, 100, 120]
OTHER_SIGNS = ['stop', 'no_left_turn', 'no_overtake', 'yield']

def compare_frame(gt, pred):
    """返回 (exact, 属性幻觉, 感知幻觉, 漏报)。同类型优先匹配，剩下的才算幻觉。"""
    pool = list(gt)
    exact = attr = hal = 0
    for p in pred:
        m = next((g for g in pool if g == p), None)                 # 完全匹配
        if m is not None:
            pool.remove(m); exact += 1; continue
        m = next((g for g in pool if g[0] == p[0]), None)           # 类型对、数值错
        if m is not None:
            pool.remove(m); attr += 1; continue
        hal += 1                                                    # GT 里连类型都没有
    return exact, attr, hal, len(pool)

def report_metrics(gts, preds):
    E = A = Hl = O = 0
    for g, p in zip(gts, preds):
        e, a, h, o = compare_frame(g, p)
        E += e; A += a; Hl += h; O += o
    n_pred, n_gt = E + A + Hl, E + A + O
    return dict(n_gt=n_gt, n_pred=n_pred,
                exact_rate=E / max(n_pred, 1),
                attr_rate=A / max(n_pred, 1),
                halluc_rate=Hl / max(n_pred, 1),
                omission_rate=O / max(n_gt, 1))

# —— 合成 800 帧 GT：**一半的帧视野里没有任何标志**（负样本必须有）——
r3 = np.random.default_rng(11)
N_FRAME = 800
GT = []
for _ in range(N_FRAME):
    f = []
    if r3.random() >= 0.50:
        if r3.random() < 0.60:
            f.append(('speed_limit', int(r3.choice(SPEED_VALUES))))
        if r3.random() < 0.45:
            f.append((str(r3.choice(OTHER_SIGNS)), None))
    GT.append(f)

def model_open(frame, r, p_ok=0.88, p_attr=0.07, p_hal=0.02):
    """探针 A（开放式）：「列出视野内所有标志，没有就输出空列表」"""
    out = []
    for s in frame:
        u = r.random()
        if u < p_ok:
            out.append(s)
        elif u < p_ok + p_attr:                                     # 属性幻觉
            if s[0] == 'speed_limit':
                out.append(('speed_limit', int(r.choice([v for v in SPEED_VALUES if v != s[1]]))))
            else:
                out.append((str(r.choice([o for o in OTHER_SIGNS if o != s[0]])), None))
        # else: 漏报
    if r.random() < p_hal:                                          # 感知幻觉
        out.append(('speed_limit', int(r.choice(SPEED_VALUES))) if r.random() < 0.6
                   else (str(r.choice(OTHER_SIGNS)), None))
    return out

def model_leading(frame, r):
    """探针 B（诱导式）：「前方限速是多少？」—— **预设了存在一块限速牌**"""
    out = model_open(frame, r)
    if not any(s[0] == 'speed_limit' for s in out) and r.random() < 0.85:
        out.append(('speed_limit', int(r.choice([50, 60, 80]))))    # 编一个「合理」的数字
    return out

rA, rB = np.random.default_rng(101), np.random.default_rng(101)
PRED_A = [model_open(f, rA) for f in GT]
PRED_B = [model_leading(f, rB) for f in GT]
mA, mB = report_metrics(GT, PRED_A), report_metrics(GT, PRED_B)

print(f"{'':<22s}{'探针A 开放式':>14s}{'探针B 诱导式':>14s}")
for k, label in [('n_pred', '报告的标志总数'), ('exact_rate', '完全正确率'),
                 ('attr_rate', '属性幻觉率'), ('halluc_rate', '**感知幻觉率**'),
                 ('omission_rate', '漏报率')]:
    fa = ('%14d' % mA[k]) if k == 'n_pred' else ('%13.1f%%' % (100 * mA[k]))
    fb = ('%14d' % mB[k]) if k == 'n_pred' else ('%13.1f%%' % (100 * mB[k]))
    print('%-22s%s%s' % (label, fa, fb))
print('\n仅仅把问题从「有哪些标志」换成「前方限速是多少」，'
      '感知幻觉率从 %.1f%% -> **%.1f%%**（%.0f 倍）'
      % (100 * mA['halluc_rate'], 100 * mB['halluc_rate'],
         mB['halluc_rate'] / mA['halluc_rate']))
assert mA['halluc_rate'] < 0.08
assert mB['halluc_rate'] > 5 * mA['halluc_rate']
assert mB['omission_rate'] <= mA['omission_rate']
print('\n⚠️ 注意一个反直觉的现象：**探针 B 的漏报率反而更低**（%.1f%% vs %.1f%%）——'
      % (100 * mB['omission_rate'], 100 * mA['omission_rate']))
print('   因为瞎猜出来的限速值偶尔会蒙对，把「漏报」变成了「命中」。')
print('   **只看召回率或 F1，你会认为诱导式探针更好** —— 这正是它危险的地方。')
print('✅ 幻觉率与漏报率必须分开报：前者要收紧生成，后者要提召回，合成 F1 等于丢掉全部决策信息。')

In [ ]:
# —— 置信度校准：下游的保守化策略全靠它 ——
def ece(conf, correct, n_bins=10):
    conf, correct = np.asarray(conf, float), np.asarray(correct, float)
    edges, e, n = np.linspace(0, 1, n_bins + 1), 0.0, conf.size
    for i in range(n_bins):
        m = (conf >= edges[i]) & (conf <= edges[i + 1]) if i == 0 else \
            (conf > edges[i]) & (conf <= edges[i + 1])
        if m.sum():
            e += m.sum() / n * abs(correct[m].mean() - conf[m].mean())
    return float(e)

r6 = np.random.default_rng(31)
N = 6000
p_true = np.clip(r6.beta(5, 2, N), 0.02, 0.98)          # 该样本真实的正确概率
correct = r6.random(N) < p_true
conf_raw = np.clip(p_true + 0.15, 0.0, 0.99)            # **语言模型的系统性过度自信**
conf_cal = np.clip(conf_raw - 0.15, 0.0, 1.0)           # 最朴素的重校准（偏移）

print(f"{'置信度区间':<14s}{'样本数':>8s}{'平均自称把握':>14s}{'实际正确率':>12s}{'差距':>9s}")
for lo in np.arange(0.5, 1.0, 0.1):
    m = (conf_raw > lo) & (conf_raw <= lo + 0.1)
    if m.sum():
        print('(%.1f, %.1f]%8d%14.3f%12.3f%9.3f'
              % (lo, lo + 0.1, m.sum(), conf_raw[m].mean(), correct[m].mean(),
                 conf_raw[m].mean() - correct[m].mean()))

e_raw, e_cal = ece(conf_raw, correct), ece(conf_cal, correct)
acc = float(correct.mean())
print('\n准确率 %.3f      ECE(原始) = %.4f      ECE(重校准后) = %.4f' % (acc, e_raw, e_cal))
assert e_raw > 0.10 and e_cal < 0.04 and e_cal < e_raw / 3
print('⚠️ ECE=%.2f 意味着模型说 0.90 的那批其实只有 ~%.2f 的正确率 ——'
      % (e_raw, 0.90 - e_raw))
print('   模块 04 里所有基于置信度的保守化阈值全部偏了。')
print('✅ 但 ECE 必须与准确率一起看：一个永远输出 0.5 且实际就是 50% 的模型 ECE=0，却毫无用处。')

## 4 · 「多少里程才能证明安全」：一个把路测这条路堵死的算术

把致命事故当泊松过程，单样本检验所需曝光量：

$$n = \frac{\bigl(z_{\alpha}\sqrt{\lambda_0} + z_{\beta}\sqrt{\lambda_1}\bigr)^{2}}{(\lambda_0-\lambda_1)^{2}}$$

$\lambda_0$ = 人类致命事故率 ≈ **1.09 起 / 亿英里**。

In [ ]:
Z_ALPHA, Z_BETA = 1.6449, 0.8416        # 单侧 alpha=0.05, power=80%

def miles_needed(lam0, lam1, z_a=Z_ALPHA, z_b=Z_BETA):
    return (z_a * math.sqrt(lam0) + z_b * math.sqrt(lam1)) ** 2 / (lam0 - lam1) ** 2

LAM0 = 1.09e-8                           # 起/英里
print(f"{'要证明比人类好':>14s}{'目标事故率(起/亿英里)':>22s}{'所需里程(亿英里)':>18s}")
for imp in (0.10, 0.20, 0.50, 0.90):
    lam1 = LAM0 * (1 - imp)
    n = miles_needed(LAM0, lam1)
    print('%13.0f%%%22.3f%18.1f' % (100 * imp, lam1 * 1e8, n / 1e8))

N20 = miles_needed(LAM0, LAM0 * 0.8)
print('\n要以 95%% 置信、80%% 功效证明**比人类好 20%%**，需要 %.1f 亿英里'
      '（RAND 的经典估算约 110 亿，量级一致）' % (N20 / 1e8))
assert 1.0e10 < N20 < 1.6e10

print(f"\n{'车队规模':<26s}{'单车日均英里':>14s}{'需要多少年':>12s}")
for fleet, mpd, label in [(100, 200, '典型测试车队'), (10_000, 100, '大型测试车队'),
                          (1_000_000, 30, '量产车队（影子模式）')]:
    years = N20 / (fleet * mpd * 365)
    print('%-26s%14d%11.1f 年' % ('%s (%d 辆)' % (label, fleet), mpd, years))
assert N20 / (100 * 200 * 365) > 1000, '100 辆车的测试车队要跑一千年以上'
assert N20 / (1_000_000 * 30 * 365) < 3, '只有量产车队规模才够 —— 但那时车已经卖出去了'
print('\n⚠️ 循环论证：要靠里程证明安全需要百万级车队；要卖出百万辆你得先证明它安全。')
print('✅ 结论：**里程用来发现问题，不用来证明安全。**安全论证必须分层：')
print('   ① 形式化保证安全层性质  ② 场景化覆盖已知失效模式')
print('   ③ 大规模闭环仿真做回归  ④ 路测发现「未知的未知」并向下沉淀成场景与单测')

## 5 · 蒸馏一致性：整体 94%，长尾 62%

云端教师（大模型）→ 车端学生（小模型）。三个数字必须分别报：
**整体一致率 / 分桶一致率 / 动作分布距离**。

而且要牢记：**一致性 ≠ 正确性** —— 完美复制教师错误的学生一致率是 100%。

In [ ]:
r5 = np.random.default_rng(21)
N_S = 20000
is_tail = r5.random(N_S) < 0.10                       # 10% 长尾场景

acc_T   = np.where(is_tail, 0.80, 0.97)               # 教师在各桶的准确率
agree_p = np.where(is_tail, 0.62, 0.98)               # 学生与教师的一致率

teacher_ok = r5.random(N_S) < acc_T
agree      = r5.random(N_S) < agree_p
student_ok = np.where(agree, teacher_ok, ~teacher_ok) # 二元决策：不一致 = 结论相反

def bucket_report(mask):
    return dict(n=int(mask.sum()), agree=float(agree[mask].mean()),
                t_acc=float(teacher_ok[mask].mean()), s_acc=float(student_ok[mask].mean()))

print(f"{'切片':<16s}{'样本数':>8s}{'一致率':>10s}{'教师准确率':>12s}{'学生准确率':>12s}")
for name, m in [('**整体**', np.ones(N_S, bool)), ('头部 (90%)', ~is_tail), ('**长尾 (10%)**', is_tail)]:
    b = bucket_report(m)
    print('%-16s%8d%9.1f%%%11.1f%%%11.1f%%' % (name, b['n'], 100 * b['agree'],
                                               100 * b['t_acc'], 100 * b['s_acc']))

ov, tl = bucket_report(np.ones(N_S, bool)), bucket_report(is_tail)
print('\n⚠️ 整体一致率 %.1f%% 看起来很健康，但它被 90%% 的头部场景主导 ——'
      '长尾桶只有 %.1f%%。' % (100 * ov['agree'], 100 * tl['agree']))
print('   学生在长尾上的准确率 %.1f%%，比教师低了 %.1f 个点。'
      % (100 * tl['s_acc'], 100 * (tl['t_acc'] - tl['s_acc'])))
assert ov['agree'] > 0.93 and tl['agree'] < 0.70
assert tl['t_acc'] - tl['s_acc'] > 0.15

# —— 一致性 != 正确性：一个 100% 一致的学生 ——
perfect_student_ok = teacher_ok.copy()
print('\n一个与教师 **100%% 一致** 的学生：一致率 100.0%%，准确率 %.1f%% ——'
      % (100 * perfect_student_ok.mean()))
print('   它完美复制了教师的全部错误。**教师的准确率就是纯蒸馏学生的天花板。**')
assert abs(perfect_student_ok.mean() - teacher_ok.mean()) < 1e-12
print('✅ 一致性只回答「学到了教师的东西吗」，正确性必须用独立的 GT 评测集回答。')

In [ ]:
# —— 压缩对 VLA 的特殊伤害：整体只掉几个点，长尾塌方 ——
#     （教学用示意值，量级与 LLM 量化的公开报告一致）
QUANT = [
    ('FP16 (baseline)',   0.952, 0.790, 0.041, 0.052),
    ('INT8 per-tensor',   0.941, 0.512, 0.193, 0.147),
    ('INT8 per-channel',  0.949, 0.751, 0.058, 0.061),
    ('INT4 (AWQ 类)',     0.938, 0.642, 0.101, 0.093),
]
HEAD_W, TAIL_W = 0.90, 0.10

print(f"{'方案':<20s}{'头部准确率':>11s}{'长尾准确率':>11s}{'加权总准确率':>13s}"
      f"{'幻觉率':>9s}{'ECE':>8s}")
base = None
for name, ha, ta, hal, e in QUANT:
    ov = HEAD_W * ha + TAIL_W * ta
    if base is None:
        base = (ov, ta, hal)
    print('%-20s%10.1f%%%10.1f%%%12.1f%%%8.1f%%%8.3f'
          % (name, 100 * ha, 100 * ta, 100 * ov, 100 * hal, e))

_, ha_pt, ta_pt, hal_pt, _ = ('',) + QUANT[1][1:]
ov_pt = HEAD_W * ha_pt + TAIL_W * ta_pt
print('\nINT8 per-tensor 相对 FP16：')
print('  加权总准确率  -%.1f 个点   <- 常规评测集上「只掉一点点」' % (100 * (base[0] - ov_pt)))
print('  **长尾准确率  -%.1f 个点** <- 而长尾恰恰是引入 VLA 的全部理由' % (100 * (base[1] - ta_pt)))
print('  **幻觉率      x%.1f**      <- 压缩后更倾向输出高频答案：「限速 15」被「纠正」成「限速 50」'
      % (hal_pt / base[2]))
assert base[0] - ov_pt < 0.05, '整体只掉不到 5 个点'
assert base[1] - ta_pt > 0.25, '长尾却掉了 25 个点以上'
assert hal_pt > 4 * base[2]
print('\n✅ VLA 压缩的验收必须看**四组数字**：常规准确率 / 长尾准确率 / 幻觉率 / ECE。')
print('   只看第一个等于没测。而更前置的问题是：**你的长尾评测集是怎么来的？**')
print('   从日志均匀采样得到的集合，按定义就没有长尾。')

## 6 · 云端-车端任务分配

约束：
1. **安全关键任务必须在车端**（蜂窝网的尾延迟与断连无法界定 WCET）
2. 每个任务选定的延迟必须 ≤ 它自己的截止时间
3. 车端总延迟 ≤ 车端预算

目标：最大化总质量。8 个任务，穷举 $2^8=256$ 种分配。

In [ ]:
# name, t_edge_ms, q_edge, t_cloud_ms(含往返), q_cloud, deadline_ms, safety_critical
TASKS = [
    ('TSR 2D 检测',        8, 0.92, 220, 0.97,   100, True),
    ('约束层（模块04）',    1, 1.00, 200, 1.00,    50, True),
    ('安全层 L3',           2, 1.00, 200, 1.00,    30, True),
    ('VLA 轨迹生成',       45, 0.85, 350, 0.96,   500, True),
    ('电子牌 OCR 兜底',    20, 0.80, 250, 0.95,   200, False),
    ('场景语义打标',       30, 0.72, 400, 0.95,  5000, False),
    ('长尾难例分析',       60, 0.60, 800, 0.98, 60000, False),
    ('自然语言解释生成',   25, 0.70, 300, 0.94,  1000, False),
]

def assign_tasks(tasks, edge_budget_ms):
    """穷举所有「车端/云端」分配，返回 (最优分配, 总质量)；不可行返回 (None, None)。"""
    best, best_q = None, -1.0
    for bits in itertools.product([0, 1], repeat=len(tasks)):     # 1 = 放车端
        edge_ms, q, ok = 0.0, 0.0, True
        for b, (name, te, qe, tc, qc, dl, crit) in zip(bits, tasks):
            if crit and not b:                    # ① 安全关键必须在车端
                ok = False; break
            lat, qual = (te, qe) if b else (tc, qc)
            if lat > dl:                          # ② 违反截止时间
                ok = False; break
            edge_ms += te if b else 0.0
            q += qual
        if ok and edge_ms <= edge_budget_ms and q > best_q:
            best, best_q = bits, q
    return best, (best_q if best else None)

for budget in (100, 80, 70):
    bits, q = assign_tasks(TASKS, budget)
    print('车端延迟预算 %d ms:' % budget, end=' ')
    if bits is None:
        print('**无可行解**')
        continue
    edge_ms = sum(t[1] for b, t in zip(bits, TASKS) if b)
    print('总质量 %.2f，车端占用 %.0f ms' % (q, edge_ms))
    for b, t in zip(bits, TASKS):
        print('    %-20s -> %-4s  (%s%s)' % (t[0], '车端' if b else '云端',
              '安全关键' if t[6] else '截止 %d ms' % t[5],
              '，云端会超时' if (not t[6] and t[3] > t[5]) else ''))

bits100, q100 = assign_tasks(TASKS, 100)
assert bits100 == (1, 1, 1, 1, 1, 0, 0, 0), bits100
assert abs(q100 - (0.92 + 1.00 + 1.00 + 0.85 + 0.80 + 0.95 + 0.98 + 0.94)) < 1e-9
assert assign_tasks(TASKS, 70)[0] is None, '预算 70 ms 时连必须放车端的任务都装不下'
print('\n✅ 两条硬规则决定了大部分答案：')
print('   ① 安全关键 -> 必须车端（不是因为云端慢，是因为它的**最坏延迟没有上界**）')
print('   ② 云端更准但更慢 -> 只有截止时间宽裕的任务才能上云；OCR 兜底被 200 ms 的截止逼回车端')

## ✏️ 练习 1：误差累积与放大倍数

实现两个函数：
1. `closed_err(eps, kappa, T)`：$\kappa>0$ 时返回 $\varepsilon\,\frac{(1+\kappa)^T-1}{\kappa}$；$\kappa=0$ 时返回 $\varepsilon T$
2. `steps_to_amplify(kappa, factor)`：**最小**的 $T$，使得「闭环误差 / 单步开环误差」$\ge$ `factor`
   （放大倍数 = $\frac{(1+\kappa)^T-1}{\kappa}$，与 $\varepsilon$ 无关；$\kappa=0$ 时放大倍数就是 $T$）

In [ ]:
def closed_err(eps, kappa, T):
    # TODO
    raise NotImplementedError

def steps_to_amplify(kappa, factor):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测（手算）——
assert abs(closed_err(0.05, 0.0, 30) - 1.5) < 1e-12          # 线性累积：0.05 * 30
assert abs(closed_err(0.05, 0.10, 1) - 0.05) < 1e-12         # 一步 = eps
assert abs(closed_err(0.05, 0.10, 30) - 8.2247) < 1e-3       # (1.1^30-1)/0.1 * 0.05
assert abs(closed_err(0.05, 0.10, 30) / 0.05 - 164.494) < 1e-2

assert steps_to_amplify(0.0, 10) == 10                       # kappa=0 -> 放大倍数就是 T
assert steps_to_amplify(0.10, 10) == 8, steps_to_amplify(0.10, 10)   # T=7 -> 9.487; T=8 -> 11.436
assert steps_to_amplify(0.05, 10) == 9, steps_to_amplify(0.05, 10)   # T=8 -> 9.549; T=9 -> 11.03
assert steps_to_amplify(0.10, 100) == 26, steps_to_amplify(0.10, 100)  # 1.1^T >= 11 -> T >= 25.16

print(f"{'kappa':>8s}{'放大 10 倍需要几步':>20s}{'放大 100 倍':>14s}{'T=30 时的放大倍数':>20s}")
for k in (0.0, 0.02, 0.05, 0.10, 0.20):
    amp30 = closed_err(1.0, k, 30)
    print('%8.2f%20d%14d%20.1f' % (k, steps_to_amplify(k, 10), steps_to_amplify(k, 100), amp30))
print('\n✅ 练习 1 通过：kappa 只要不是 0，闭环误差就是指数的 ——')
print('   而开环评测报的永远是那个不变的 eps。')

## ✏️ 练习 2：防作弊的复合分

实现 `ds(rc, infractions, penalty)`：$\text{DS}=\text{RC}\times\prod_i p_i^{n_i}$。

- `rc`：路线完成率 ∈ [0,1]
- `infractions`：`{违规类型: 次数}`
- `penalty`：`{违规类型: 折扣系数}`，缺省用 `PENALTY`

重点体会：**没有 RC 这个乘数，「停着不动」在所有违规项上都是满分。**

In [ ]:
def ds(rc, infractions, penalty=None):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测（手算）——
assert abs(ds(1.0, {}) - 1.0) < 1e-12
assert abs(ds(1.0, {'collision_vehicle': 1}) - 0.60) < 1e-12
assert abs(ds(1.0, {'collision_vehicle': 2}) - 0.36) < 1e-12
assert abs(ds(0.8, {'red_light': 1, 'speeding': 2}) - 0.8 * 0.70 * 0.81) < 1e-12
assert abs(ds(1.0, {}, {'x': 0.5}) - 1.0) < 1e-12                     # 没有违规就没有折扣

# 「停着不动」的作弊路径必须被堵死
crawl = ds(0.167, {})                                                  # 零违规，但只走了 16.7%
smooth = ds(1.0, {})                                                   # 完成 + 零违规
risky = ds(1.0, {'collision_vehicle': 1, 'speeding': 1})               # 完成但撞了 + 超速
print('极度保守（零违规，RC=0.167）: DS = %.3f' % crawl)
print('均衡  （零违规，RC=1.00 ）: DS = %.3f' % smooth)
print('激进  （1 碰撞 + 超速     ）: DS = %.3f' % risky)
assert crawl < smooth and risky < smooth
assert crawl < risky, '「停着不动」甚至不如「撞了一次但完成了路线」——这正是乘法结构想表达的'
print('\n✅ 练习 2 通过：**完成率是乘数，违规是折扣**。')
print('   任何只报安全指标的评测体系，都有一条「不开就不会错」的作弊路径。')

## ✏️ 练习 3：幻觉的四类分解

实现 `halluc_report(gts, preds)`：`gts` / `preds` 是等长的列表，每个元素是该帧的标志集合
（元素形如 `('speed_limit', 60)` 或 `('stop', None)`）。返回

```
dict(exact_rate, attr_rate, halluc_rate, omission_rate)
```

匹配规则（顺序很重要）：对每个预测，先找**完全相同**的 GT（exact），
再找**类型相同但数值不同**的 GT（属性幻觉），都没有才算**感知幻觉**；
GT 里没被匹配掉的算**漏报**。前三个的分母是预测总数，漏报的分母是 GT 总数（分母为 0 时取 1）。

In [ ]:
def halluc_report(gts, preds):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测（手算）——
g = [[('speed_limit', 60)],
     [],
     [('stop', None), ('speed_limit', 80)]]
p = [[('speed_limit', 80)],      # 类型对数值错 -> 属性幻觉
     [('speed_limit', 50)],      # GT 是空的   -> 感知幻觉
     [('stop', None)]]           # 漏了 speed_limit 80
r = halluc_report(g, p)
print(r)
assert abs(r['exact_rate']    - 1 / 3) < 1e-12, r
assert abs(r['attr_rate']     - 1 / 3) < 1e-12, r
assert abs(r['halluc_rate']   - 1 / 3) < 1e-12, r
assert abs(r['omission_rate'] - 1 / 3) < 1e-12, r

# 全部漏报：幻觉率 0，漏报率 1 —— 一个「什么都不说」的模型幻觉率完美
r2 = halluc_report(g, [[], [], []])
assert r2['halluc_rate'] == 0.0 and abs(r2['omission_rate'] - 1.0) < 1e-12
print('全部漏报 ->', r2)
print('⚠️ 「什么都不说」的模型幻觉率是 0 —— **幻觉率单独看同样可以被作弊**，必须与漏报率成对报。')

# 全部正确
r3 = halluc_report(g, g)
assert abs(r3['exact_rate'] - 1.0) < 1e-12 and r3['omission_rate'] == 0.0
# 用第 3 节的 800 帧数据复现主结论
rr = halluc_report(GT, PRED_B)
assert rr['halluc_rate'] > 5 * halluc_report(GT, PRED_A)['halluc_rate']
print('✅ 练习 3 通过：四类分解 + 成对报告，才是能驱动修复动作的度量。')

## ✏️ 练习 4：云端-车端分配求解器

实现 `assign(tasks, edge_budget_ms)`，`tasks` 的每一项是
`(name, t_edge, q_edge, t_cloud, q_cloud, deadline, safety_critical)`。

约束：① 安全关键必须在车端；② 选定方案的延迟 ≤ 该任务的截止时间；
③ 车端任务的延迟之和 ≤ `edge_budget_ms`。目标：最大化总质量。

返回 `(bits, total_quality)`，`bits` 是 0/1 元组（1 = 车端）；无可行解返回 `(None, None)`。

In [ ]:
def assign(tasks, edge_budget_ms):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测（3 个任务，可手算）——
T3 = [('A', 10, 0.90, 100, 0.95,   50, True),    # 安全关键 -> 必须车端
      ('B', 20, 0.70, 150, 0.90,  100, False),   # 云端 150 ms > 截止 100 ms -> 只能车端
      ('C', 30, 0.60, 200, 0.95, 5000, False)]   # 截止宽裕 -> 上云更优

bits, q = assign(T3, 40)
assert bits == (1, 1, 0), bits
assert abs(q - (0.90 + 0.70 + 0.95)) < 1e-9, q
print('预算 40 ms -> 分配 %s，总质量 %.2f，车端占用 %d ms' % (bits, q, 10 + 20))

assert assign(T3, 25) == (None, None), '必须车端的 A+B 就要 30 ms，装不下'
assert assign(T3, 1000)[0] == (1, 1, 0), '预算再大也不会把 C 拉回车端 —— 云端质量更高'

# 把 C 的截止时间收紧到 150 ms -> 云端 200 ms 超时 -> C 被逼回车端
T3b = list(T3); T3b[2] = ('C', 30, 0.60, 200, 0.95, 150, False)
bits_b, q_b = assign(T3b, 100)
assert bits_b == (1, 1, 1) and abs(q_b - (0.90 + 0.70 + 0.60)) < 1e-9, (bits_b, q_b)
print('把 C 的截止收紧到 150 ms -> 分配 %s，总质量掉到 %.2f' % (bits_b, q_b))

# 用第 6 节的 8 任务实例复现
assert assign(TASKS, 100) == assign_tasks(TASKS, 100)
print('✅ 练习 4 通过：截止时间才是真正的约束 —— 「上云更准」在硬实时任务上根本用不上。')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def closed_err(eps, kappa, T):
    return eps * T if kappa == 0 else eps * ((1 + kappa) ** T - 1) / kappa

def steps_to_amplify(kappa, factor):
    T = 1
    while closed_err(1.0, kappa, T) < factor:
        T += 1
    return T

In [ ]:
# 练习 2 参考答案
def ds(rc, infractions, penalty=None):
    p = penalty or PENALTY
    out = float(rc)
    for k, n in infractions.items():
        out *= p[k] ** n
    return out

In [ ]:
# 练习 3 参考答案
def halluc_report(gts, preds):
    E = A = Hl = O = 0
    for gt, pred in zip(gts, preds):
        pool = list(gt)
        for q in pred:
            m = next((x for x in pool if x == q), None)
            if m is not None:
                pool.remove(m); E += 1; continue
            m = next((x for x in pool if x[0] == q[0]), None)
            if m is not None:
                pool.remove(m); A += 1; continue
            Hl += 1
        O += len(pool)
    n_pred, n_gt = max(E + A + Hl, 1), max(E + A + O, 1)
    return dict(exact_rate=E / n_pred, attr_rate=A / n_pred,
                halluc_rate=Hl / n_pred, omission_rate=O / n_gt)

In [ ]:
# 练习 4 参考答案
def assign(tasks, edge_budget_ms):
    best, best_q = None, -1.0
    for bits in itertools.product([0, 1], repeat=len(tasks)):
        edge_ms, q, ok = 0.0, 0.0, True
        for b, (name, te, qe, tc, qc, dl, crit) in zip(bits, tasks):
            if crit and not b:
                ok = False; break
            lat, qual = (te, qe) if b else (tc, qc)
            if lat > dl:
                ok = False; break
            edge_ms += te if b else 0.0
            q += qual
        if ok and edge_ms <= edge_budget_ms and q > best_q:
            best, best_q = bits, q
    return (best, best_q) if best else (None, None)

---
## 🧪 真实工程胶囊：VLA 评测与上车的完整检查单

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# VLA 评测与上车 · 检查单（按顺序做，每一步都有通过条件）
# ══════════════════════════════════════════════════════════════════════

# ① 先做一次「开环-闭环相关性验证」（**只做一次，但决定之后所有实验的可信度**）
#    取 8-10 个能力差异明显的模型 -> 同时算开环 ADE/FDE 与闭环 Driving Score
#    算 Spearman rho；画散点图
#    通过条件：rho >= 0.5 才允许把开环指标用于**选型**；否则它只能做监控信号
#    必做的对照：**ego-state ablation** —— 去掉自车历史输入，看指标掉多少
#      再加一个「纯常速外推」的盲基线；若盲基线 >= 你模型 90% 的分数，
#      这个 benchmark 在这个任务上没有分辨力，别再往上刷了

# ② 指标看板（六类，缺一类就有一条作弊路径）
#    安全 : collisions/1000km（**做 at-fault 判定**）、TTC<1.5s 次数
#    合规 : overspeed_distance_ratio（分母 = **该限速约束 ACTIVE 的里程**）
#           sign_compliance_rate（分母 = 遇到的**适用**标志数，排除对向/货车/时间窗外）
#           over_hold_p95_m、early_release_count（**硬门禁 = 0**）
#    接管 : MPI、critical_intervention_rate（关键 / 非关键**必须分开**）
#    舒适 : |a|_p95、jerk_p95（**用分位数不用均值**）
#    效率 : unnecessary_brakes / 100km（过度保守的度量）
#    进度 : route_completion（**没有它，停着不动是满分**）
#    复合 : DS = RC * prod(p_i ^ n_i)     <- 乘法，不是加权和

# ③ 幻觉与校准（VLA 独有，检测器时代没有的一栏）
#    探针必须开放式且**允许输出空列表**；禁止「前方限速是多少」这种预设式提问
#    评测集必须含：**无标志帧**（负样本）、罕见标志帧、时序连续帧（测粘性幻觉）
#    报四个数：exact / attr_halluc / **perception_halluc** / omission —— **不要合成 F1**
#    另报 ECE（与准确率一起看）；置信度用 token logprob 或多次采样一致性，别让模型自己说

# ④ 场景库与覆盖率
#    组织：与失效模式一一对应（C55 m03 的全景图，每项一组场景）
#    分层：功能场景 -> 逻辑场景（参数化）-> 具体场景（参数取定）
#    报两种覆盖率：
#      参数空间覆盖（分层采样 + 危险区域重点采）
#      **约束状态机转移覆盖**（模块 04 的 6 组转移边，每条至少触发一次）
#        尤其是 ROAD_CLASS_CHANGE 导致的 EXPIRED 与 SUPERSEDED —— 最难复现的线上 bug
#    门禁：旧场景零回退（硬）；多重比较校正；**阈值提前冻结**

# ⑤ 可测试性（**必须在架构阶段定，事后补不回来**）
#    三个有名字、可独立判定对错的中间对象：
#      结构化感知输出（+置信度）-> 约束集（+消解 trace）-> 安全层判决（+原因）
#    归因表：感知对错 x 约束对错 x VLA 对错 x 安全层是否拦住 -> 5 种根因、5 种修复动作
#    日志：常态 10Hz 结构化 DecisionRecord；事件触发存前后 10s 完整数据
#          触发条件 = 安全层 REJECT / 接管 / 急刹 / 消解平票 / 约束状态异常跳变
#    **schema 版本化 + 向后兼容解析器**；每条日志记模型版本/约束层版本/数据版本

# ⑥ 云端-车端与压缩
#    分工判据：**任何进入安全关键回路的计算必须在车端**（云端最坏延迟无上界）
#    蒸馏一致性报三个数：整体一致率 / **按稀有度分桶的一致率** / 动作分布距离
#      牢记：一致性 != 正确性；教师准确率是纯蒸馏学生的天花板
#    压缩验收看四组数：常规准确率 / **长尾准确率** / **幻觉率** / ECE
#      Transformer 的激活离群点 -> per-tensor INT8 会崩，用 per-channel / SmoothQuant / AWQ
#      KV cache 要一起量化，误差沿序列累积

# ⑦ 上车分阶段（每阶段的准入/退出判据**提前定义并冻结**）
#    阶段0 仿真闭环   -> 阶段1 影子模式 -> 阶段2 有限 ODD -> 阶段3 扩 ODD -> 阶段4 全量
#    影子模式能测：漏检/误检、**幻觉率**、约束集正确性、罕见场景发现
#    影子模式**不能**测：闭环稳定性 —— 车是人在开，模型输出没改变状态，结构上仍是开环
#    分歧要做三分类：模型错 / 人错 / 都可以；先用约束层自动滤掉人类明显违规的片段
#    「高置信度分歧」是最有价值的子集
#    **OTA 回滚能力本身要当作可靠性需求来验证，并定期演练**

# ⑧ 上线后的持续监控（这四个是「免费传感器」，不需要额外标注）
#    intervention_rate / halluc_rate / tie_rate / over_hold_p95
#    任何一个突变都报警；回滚条件写成可自动判定的规则
'''
print(RECIPE)
for token in ['ego-state ablation', 'route_completion', 'early_release_count',
              'perception_halluc', '约束状态机转移覆盖', 'schema 版本化',
              '长尾准确率', '仍是开环', 'OTA 回滚']:
    assert token in RECIPE, token
print('✅ 检查单覆盖：相关性验证 / 六类指标 / 幻觉与校准 / 场景覆盖 / 可测试性 / 云端与压缩 / 分阶段上车 / 持续监控')

### 小结

- **开环评测系统性高估，而且高估的方式有三种**：误差累积（$e_T=\varepsilon\frac{(1+\kappa)^T-1}{\kappa}$，
  $\kappa=0.1,T=30$ 时放大 **164 倍**）、**自车状态捷径**（不看图像的常速外推在整体 ADE 上赢 14%，
  在弯道桶上差 **18 倍**）、多模态惩罚（还会把模型逼成输出两个模式的平均）。
  notebook 里两个开环 ADE 相差 4% 的模型，闭环失败率差 **81 倍**。
  **正确定位：开环是便宜的负向筛子，不能用于选型。上手先做一次开环-闭环相关性验证。**
- **指标必须成对，而且是乘法。**六类：安全 / 合规 / 接管 / 舒适 / 效率 / 进度。
  没有进度类指标，「停在原地」在所有安全指标上都是满分；
  CARLA 的 $\text{DS}=\text{RC}\times\prod p_i^{n_i}$ 就是为了堵死这条路。
  **TSR 专项指标要盯分母**：超速率的分母必须是「该限速 ACTIVE 的里程」，
  否则「约束提前失效」这个 bug 反而会降低超速率——**指标把故障掩盖了**。
- **幻觉是 VLA 新增的失效模式**，要分成感知 / 属性 / 推理三种，并与漏报**成对**报告。
  探针不能有预设：把「有哪些标志」换成「前方限速是多少」，感知幻觉率从 5.6% → **55.7%**，
  而**漏报率反而下降**（瞎猜偶尔蒙对）——只看 F1 会得出反的结论。
  评测集必须含无标志帧；置信度要报 ECE 且与准确率一起看。
- **路测里程在数学上无法证明安全**：要以 95% 置信、80% 功效证明比人类好 20%，
  需要 **132 亿英里**；100 辆车的测试车队要跑 1800 年。
  **里程用来发现问题，安全论证靠分层**：形式化 + 场景化 + 闭环仿真 + 路测发现未知。
- **可测试性只能提前规划**：三个有名字、可独立判定的中间对象（感知输出 / 约束集 / 安全层判决），
  把归因深度从 $\infty$ 降到 $O(\log L)$。日志要 schema 版本化，否则三个月前的数据会变成垃圾。
- **蒸馏一致性要分桶**：整体 94.6% 而长尾桶只有 63%；而且**一致性 ≠ 正确性**，
  教师准确率是纯蒸馏学生的天花板。**压缩对 VLA 的伤害集中在长尾**：
  INT8 per-tensor 整体只掉 3.8 点，长尾掉 **27.8 点**，幻觉率涨 **4.7 倍**。
- **影子模式在结构上仍是开环**：车是人在开，模型输出没有改变状态。
  它是感知与语义层的大规模验证器，不是闭环安全的证明。分歧 ≠ 错误，要做三分类。

至此 C59 全部结束。回头看这门课的主线：**VLA 把长尾语义理解能力接进了自动驾驶，
但它换来的每一分能力，都必须配一套显式的接口（m03）、一层可验证的约束与兜底（m04）、
和一套防作弊的评测（m05）——否则那分能力在量产系统里是不可交付的。**